In [ ]:
from __future__ import annotations

from typing import List, Tuple, Iterable
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

def load_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def preprocess(df: pd.DataFrame) -> Tuple[np.ndarray, List[str], pd.DataFrame]:
    
    drop_cols = ["Customer Id", "Defaulted", "Address"]
    df = df.drop(columns=drop_cols)
    df_num = df.select_dtypes(include= np.number)
    df_clean = df_num.dropna()

    scaler = StandardScaler()
    X_scaler = scaler.fit_transform(df_clean)

    feature_names = df_clean.columns.tolist()

    return X_scaler,feature_names,df_clean
 
def elbow_inertia(X: np.ndarray, k_min: int = 1, k_max: int = 10, random_state: int = 42) -> List[float]:
   
    inertias = []
    for k in range(k_min,k_max+1):
        km =KMeans(n_clusters=k,random_state=random_state)
        km.fit(X)
        inertias.append(km.inertia_)
    global _L_inertias
    _L_inertias = inertias
    return inertias   

def identify_elbow_k() -> int:
    
    inertias = _L_inertias
    d1 = np.diff(inertias)
    d2 = np.diff(d1)
    k = int(np.argmax(d2)) +2
    return k
 
def kmeans_cluster(X: np.ndarray, n_clusters: int, random_state: int = 42) -> Tuple[np.ndarray, KMeans]:
   
    km = KMeans(n_clusters=n_clusters,random_state=random_state)
    labels = km.fit_predict(X)
    return labels,km
    
def kmeans_add_labels_and_centroids(
    df_clean: pd.DataFrame,
    labels: np.ndarray,
    feature_names: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    df_with_labels = df_clean.copy()
    df_with_labels['cluster_kmeans'] = labels

    centroids_df = df_with_labels.groupby('cluster_kmeans')[feature_names].mean()
    return df_with_labels,centroids_df

def _count_clusters(labels: np.ndarray) -> int:
    
    uniq = set(labels.tolist())
    if -1 in uniq:
        uniq.remove(-1)
    return len(uniq)

def dbscan_cluster_to_target_k(
    X: np.ndarray,
    target_k: int = 1,
) -> Tuple[np.ndarray, DBSCAN]:
    
    eps_grid = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 2.0])
    min_samples_grid = np.array([3, 5, 8])
    for eps in eps_grid:
        for min_samples in min_samples_grid:
            model = DBSCAN(eps=eps,min_samples=min_samples,metric='euclidean')
            labels = model.fit_predict(X)
            if _count_clusters(labels) == target_k:
                return labels, model
    return labels,model


def dbscan_add_labels(df_clean: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
    
    df_with_cluster_dbscan = df_clean.copy()
    df_with_cluster_dbscan['cluster_dbscan'] = labels
    return df_with_cluster_dbscan   

def compute_silhouettes(
    X: np.ndarray,
    km_labels: np.ndarray,
    db_labels: np.ndarray,
) -> Tuple[float, float]:
    
    km_sil = float(silhouette_score(X,km_labels))

    mask = db_labels != -1
    n_cluster = len(set(db_labels)) -(1 if -1 in db_labels else 0)
    n_noise = list(db_labels).count(-1)
    n_remain = mask.sum()

    if n_cluster >=2 and n_remain >=2:
        db_sil = float(silhouette_score(X[mask],db_labels[mask],metric='euclidean'))
    else:
        db_sil = float('nan')
    return float(km_sil),float(db_sil)
    # TODO: implement
    raise NotImplementedError("Implement compute_silhouettes(X, km_labels, db_labels)")